# LLM Agent and Tools


### 1. Agent
An agent is an AI system that can decide what to do, call a tool if required and iterates until the task is complete.

- Decision making capibility
- Has reasoning ability
- call multiple tools if required
- talk to other agents 

### 2. Tool: 
A tool does one defined job when called.

- No decision-making ability 
- Controlled by the model or agent
- Single Responsibility
- Has a fixed interface (inputs → outputs)



In [ ]:
!uv pip install psycopg2-binary sqlalchemy pandas python-dotenv openai

In [ ]:
from sqlalchemy import create_engine, text

# Panda
import pandas as pd

# Dotenv
from dotenv import load_dotenv
import os

from IPython.display import display, Markdown

from openai import OpenAI

# Parse JSON response
import json
# Load environment variables from .env file
load_dotenv(override=True)



In [ ]:



GROK_API_KEY = os.getenv("GROK_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL") # grok-4-1-fast-reasoning
# print(f"Grok API Key: {GROK_API_KEY}")


# Initialize Grok API client
# You can use xAI's Grok API through OpenAI-compatible interface
client = OpenAI(
    api_key=GROK_API_KEY,  # Replace with your actual Grok API key
    base_url="https://api.x.ai/v1"
)

In [ ]:
# Database connection
DATABASE_URL = os.getenv("DATABASE_URL")
print(f"Connecting to database at: {DATABASE_URL}")

engine = create_engine(DATABASE_URL)

#### 1. Create a simple tool ```get_info_subhadip_ghorui_tool``` 
To fetch information about Subhadip Ghorui from a context file and return it as a string. 

This will be used to test the tool integration with the LLM.

In [ ]:
def get_info_subhadip_ghorui_tool():
    print("Tool called: get_info_subhadip_ghorui_tool")
     # Read from context memory file
    file_path = os.path.join(os.getcwd(), "..", "context", "Subhadip Ghorui.md")
    with open(file_path, "r", encoding="utf-8") as f:
        data = f.read()
    return data


# tool context here ...
get_info_subhadip_ghorui_tool_context = {
        "type": "function",
        "function": {
            "name": "get_info_subhadip_ghorui_tool",
            "description": "Get profile and background information about Subhadip Ghorui, including skills, projects, and experience.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }

### 2. Query student db using user prompt ```query_student_db_tool``` 

In [ ]:

def query_student_db_tool(question: str) -> str:
    print(f"Tool called: query_student_db_tool\nQuestion: {question}")

    SCHEMA_INFO = """
    Database Schema:

    1. classes table: id, name, status
    2. students table: id, first_name, last_name, gender, dob, age, class_id
    3. subjects table: id, name, class_id, description
    4. test_marks table: id, name, class_id, student_id, subject_id, marks, total_marks

    Relationships:
    - Student belongs to one Class (students.class_id -> classes.id)
    - Subject belongs to one Class (subjects.class_id -> classes.id)
    - TestMarks belongs to Student, Class, and Subject

    By default LIMIT result to 10000 rows if limit is not specified.
    """

    CUSTOM_INSTRUCTIONS = """
    Exceptions:
    - Subject names include the class name as a suffix (e.g., "Math Five", "Physics Six").
    If the user specifies a class, use: WHERE name = '<subject> <class>'
    If the user does not specify a class, use LIKE: WHERE name LIKE '<subject>%'
    """

    prompt = f"""
    {SCHEMA_INFO}
    {CUSTOM_INSTRUCTIONS}
    User Question: {question}

    Generate a PostgreSQL SELECT query to answer this question.
    Return ONLY a JSON object with a single key 'query'.
    """

    sql_response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a SQL expert. Generate only valid PostgreSQL SELECT queries. Return JSON format with key 'query'."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )

    response_data = json.loads(sql_response.choices[0].message.content)
    sql_query = response_data.get("query", "") or response_data.get("sql", "")

    if not sql_query:
        return "❌ Failed to generate SQL query."

   
    # Safety: only allow SELECT and disallow destructive keywords / multiple statements
    normalized = sql_query.strip()
    upper_sql = normalized.upper()

    if not upper_sql.startswith("SELECT"):
        return "❌ Only SELECT queries are allowed."

    # Disallow destructive keywords
    forbidden = ["DELETE", "UPDATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "EXEC", "MERGE"]
    found = [kw for kw in forbidden if kw in upper_sql]
    if found:
        return f"❌ Destructive queries are not allowed. Found: {', '.join(found)}"

    # Prevent multiple statements (semicolons before the end indicate additional statements)
    if ";" in upper_sql[:-1]:
        return "❌ Multiple SQL statements are not allowed."
    # print(f"Generated SQL:\n{sql_query}\n")

    with engine.connect() as conn:
        result = conn.execute(text(sql_query))
        rows = result.fetchall()
        columns = list(result.keys())

    if not rows:
        return "No records found."

    df = pd.DataFrame(rows, columns=columns)
    return f"**SQL Query:**\n```sql\n{sql_query}\n```\n\n**Result:**\n{df.to_markdown(index=False)}"


# tool context here ...
query_student_db_tool_context = {
    "type": "function",
    "function": {
        "name": "query_student_db_tool",
        "description": "Query the school student database using a natural language question. Generates a dynamic SQL SELECT query and returns the results as a markdown table.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The natural language question to query the student database."
                }
            },
            "required": ["question"]
        }
    }
}


### 3. ```update_student_mark_tool``` with student name, subject, marks required

In [ ]:

def update_student_mark_tool(first_name: str, last_name: str, subject: str, marks: float) -> str:
    print(f"Tool called: update_student_mark_tool\nStudent: {first_name} {last_name}, Subject: {subject}, Marks: {marks}")

    find_student_sql = text("""
        SELECT id, first_name, last_name FROM students
        WHERE LOWER(first_name) = LOWER(:first_name)
          AND LOWER(last_name) = LOWER(:last_name)
        LIMIT 1
    """)

    find_subject_sql = text("""
        SELECT id, name FROM subjects
        WHERE LOWER(name) LIKE LOWER(:subject_pattern)
        LIMIT 1
    """)

    update_sql = text("""
        UPDATE test_marks
        SET marks = :marks
        WHERE student_id = :student_id
          AND subject_id = :subject_id
        RETURNING id, marks
    """)

    with engine.begin() as conn:
        # Find student
        student_row = conn.execute(find_student_sql, {
            "first_name": first_name.strip(),
            "last_name": last_name.strip()
        }).fetchone()

        if not student_row:
            return f"❌ Student '{first_name} {last_name}' not found in the database."

        student_id = student_row[0]
        full_name = f"{student_row[1]} {student_row[2]}"

        # Find subject (partial match)
        subject_pattern = f"%{subject.strip()}%"
        subject_row = conn.execute(find_subject_sql, {"subject_pattern": subject_pattern}).fetchone()

        if not subject_row:
            return f"❌ Subject matching '{subject}' not found in the database."

        subject_id, subject_name = subject_row[0], subject_row[1]

        # Perform update
        result = conn.execute(update_sql, {
            "marks": marks,
            "student_id": student_id,
            "subject_id": subject_id
        })
        updated_rows = result.fetchall()

    if not updated_rows:
        return f"❌ No test marks record found for '{full_name}' in subject '{subject_name}'."

    raw_sql = f"UPDATE test_marks SET marks = {marks} WHERE student_id = {student_id} AND subject_id = {subject_id}"

    
    return (
        f"**SQL Query:**\n```sql\n{raw_sql}\n```\n\n"
        f"✅ Successfully updated marks for **{full_name}** in **{subject_name}** to **{marks}**."
    )

update_student_mark_tool_context = {
    "type": "function",
    "function": {
        "name": "update_student_mark_tool",
        "description": "Update the mark of a student for a specific subject in the school database. Looks up the student by first and last name. Returns an error if the student or subject is not found.",
        "parameters": {
            "type": "object",
            "properties": {
                "first_name": {
                    "type": "string",
                    "description": "First name of the student (e.g., 'John')."
                },
                "last_name": {
                    "type": "string",
                    "description": "Last name of the student (e.g., 'Doe')."
                },
                "subject": {
                    "type": "string",
                    "description": "Name or partial name of the subject (e.g., 'Math', 'Biology Five')."
                },
                "marks": {
                    "type": "number",
                    "description": "The new marks to set for the student in the given subject."
                }
            },
            "required": ["first_name", "last_name", "subject", "marks"]
        }
    }
}

### Main Agent

In [ ]:

def main(prompt=None):
    if not prompt:
        print("No prompt provided.")
        return None

    messages = [
        {"role": "system", "content": "You are a helpful assistant. When you query a database, always include the SQL query used in your final response as title 'RAW SQL' and formatted as a SQL code block before showing the results."},
        {"role": "user", "content": prompt}
    ]

    while True:
        print("=====API call with messages==========================")
        print(messages)
        
        try:
            response = client.chat.completions.create(
                    model=LLM_MODEL,
                    messages=messages,
                    tools=[
                        get_info_subhadip_ghorui_tool_context,
                        query_student_db_tool_context,
                        update_student_mark_tool_context],
                    temperature=0.2
                )
        except Exception as e:
            return f"Error calling Grok API: {e}"

        msg = response.choices[0].message

        print("======Response============================")
        print(msg)
    
        # final answer
        if not msg.tool_calls:
            content = msg.content or ""
            return content

        # Tool call — dispatch and feed result back
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments) if call.function.arguments else {}
            if call.function.name == "get_info_subhadip_ghorui_tool":
                result = get_info_subhadip_ghorui_tool()
            elif call.function.name == "query_student_db_tool":
                result = query_student_db_tool(**args)
            elif call.function.name == "update_student_mark_tool":
                result = update_student_mark_tool(**args)
            else:
                result = f"Tool {call.function.name} not implemented."
            
            # INPORTANT: Append tool result to messages to provide context for next LLM response    
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": result
            })
       
       
        


In [ ]:
ans = main("What is capital of India?")
display(Markdown(ans))

In [ ]:
ans = main("Tell me about Subhadip Ghorui")
display(Markdown(ans))

In [ ]:
ans = main("Get subject marks of John Doe")
display(Markdown(ans))

In [ ]:
ans = main("Update John Doe's Math mark to 88")
display(Markdown(ans))

### Gradio UI

In [ ]:
import gradio as gr

# Close previous Gradio instances to avoid port conflicts (for notebook environments)
try:
    import gradio
    gradio.close_all()
except Exception:
    pass

def chat_with_db(message, history):
    question = (message or "").strip()
    if not question:
        return history, "Please enter a question."
    try:
        answer = main(question)
    except Exception as e:
        answer = f"Error: {e}"
    history = history or []
    # Use dicts with 'role' and 'content' keys for Gradio Chatbot compatibility
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})
    return history, ""

def clear_chat():
    return [], ""

with gr.Blocks() as demo:
     # Title and subtitle
    gr.Markdown("# GenAI Chatbot")
    gr.Markdown("### Find any student details by just writing prompt and update test marks")

    chatbot = gr.Chatbot()
    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type your question...",
            lines=1,
            scale=8,
            container=False
        )
        submit_btn = gr.Button("Submit", scale=1)
    with gr.Row():
        clear_btn = gr.Button("Clear Chat", scale=9)
        
    submit_btn.click(
        fn=chat_with_db,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )
    clear_btn.click(
        fn=clear_chat,
        inputs=None,
        outputs=[chatbot, msg]
    )

demo.launch()
